# MedGemma Local Experimentation (Windows)

This notebook runs MedGemma locally using Ollama. No GPU required.

## One-Time Setup (do this ONCE before running notebook)

1. **Install Ollama**: Download from https://ollama.com/download/windows and run installer
2. **Download model**: Open Command Prompt and run:
   ```
   ollama pull dcarrascosa/medgemma-1.5-4b-it:Q4_K_M
   ```
   (This downloads ~3GB, takes a few minutes)

## Instructions

1. Run cells 1-3 to check setup
2. Edit the CONFIGURATION cell (prompt and settings)
3. Run the experiment cell
4. View results

## Cell 1: Check Ollama Installation

In [ ]:
import subprocess
import sys

try:
    result = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    print(f"[OK] Ollama installed: {result.stdout.strip()}")
except FileNotFoundError:
    print("[ERROR] Ollama not found!")
    print("Download from: https://ollama.com/download/windows")
    print("Install it, then restart this notebook")


## Cell 2: Check MedGemma Model

In [ ]:
import subprocess

MODEL = "dcarrascosa/medgemma-1.5-4b-it:Q4_K_M"

result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if "medgemma" in result.stdout.lower():
    print(f"[OK] MedGemma model ready")
else:
    print(f"[ERROR] MedGemma not found. Run this in Command Prompt:")
    print(f"  ollama pull {MODEL}")


## Cell 3: Install Python Dependencies

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ollama", "datasets", "pillow", "requests"])
print("[OK] Dependencies installed")


## Cell 4: Load Dataset

In [ ]:
from datasets import load_dataset, concatenate_datasets

print("Loading dataset (first run downloads ~500MB)...")
ds_raw = load_dataset("MartiHan/Open-MELON-VL-2.5K")
dataset = concatenate_datasets([ds_raw["train"], ds_raw["validation"], ds_raw["test"]])
print(f"[OK] Dataset loaded: {len(dataset)} images")


## Cell 5: Test Set (edit if needed)

In [ ]:
# Clean indices verified to have correct labels
NEVUS_INDICES = [33, 43, 120, 121, 170, 171, 196, 197, 205, 206]
MELANOMA_INDICES = [3, 4, 5, 6, 7, 8, 9, 20, 22, 24]
TEST_INDICES = NEVUS_INDICES + MELANOMA_INDICES

# Ground truth labels
LABELS = {i: "nevus" for i in NEVUS_INDICES}
LABELS.update({i: "melanoma" for i in MELANOMA_INDICES})

# Cleaned captions
CLEANED_CAPTIONS = {
    3: "melanoma, with features suggestive of superficial spreading melanoma.",
    4: "melanoma, with features suggestive of superficial spreading melanoma, likely in situ or early invasive.",
    5: "superficial spreading melanoma",
    6: "melanoma",
    7: "melanoma with histological features suggestive of superficial spreading melanoma.",
    8: "melanoma with morphological features suggestive of superficial spreading melanoma.",
    9: "melanoma with morphologic features suggestive of superficial spreading melanoma.",
    20: "melanoma with features suggestive of a superficial spreading melanoma subtype.",
    22: "melanoma in situ or early melanoma",
    24: "superficial spreading melanoma",
    33: "melanocytic nevus with features suggestive of a dysplastic (atypical) nevus.",
    43: "melanocytic nevus, specifically a compound nevus or junctional nevus with mild atypia.",
    120: "congenital melanocytic nevus with features suggesting a medium-sized congenital nevus.",
    121: "congenital melanocytic nevus with features suggesting a medium-sized congenital nevus.",
    170: "melanocytic nevus, likely a compound nevus given its papillomatous architecture.",
    171: "melanocytic nevus, likely a compound nevus with papillomatous features.",
    196: "melanocytic nevus (intradermal nevus with balloon cell change)",
    197: "melanocytic nevus (intradermal nevus with balloon cell change)",
    205: "melanocytic nevus, with features suggestive of an intradermal nevus with neurotization.",
    206: "melanocytic nevus, with features suggestive of an intradermal nevus with neurotization."
}

print(f"[OK] Test set: {len(NEVUS_INDICES)} nevus + {len(MELANOMA_INDICES)} melanoma = {len(TEST_INDICES)} images")


---

## CONFIGURATION (edit this cell)

Change the prompt and settings below, then run the experiment.

**Example prompts to try:**
- `"What is the diagnosis for this skin lesion image?"`
- `"Is this melanoma or a benign nevus? Answer in one word."`
- `"Classify this dermoscopic image as melanoma or nevus."`

In [ ]:
# ========== EDIT THIS ==========

PROMPT = "What is the diagnosis for this skin lesion image?"

SETTINGS = {
    "temperature": 0.0,  # 0 = deterministic, higher = more random
    "top_k": 40,         # consider top K tokens (lower = more focused)
    "top_p": 0.9,        # nucleus sampling (lower = more focused)
    "max_tokens": 150,   # max response length
}

# ===============================
print(f"Prompt: {PROMPT}")
print(f"Settings: {SETTINGS}")


## Run Experiment

In [ ]:
import ollama
import io
import base64
from PIL import Image

MODEL = "dcarrascosa/medgemma-1.5-4b-it:Q4_K_M"
results = []

def classify_response(text):
    text = text.lower()
    has_melanoma = "melanoma" in text
    has_nevus = "nevus" in text or "nevi" in text or "benign" in text
    if has_melanoma and not has_nevus:
        return "melanoma"
    elif has_nevus and not has_melanoma:
        return "nevus"
    elif has_melanoma:
        return "melanoma"
    return "unknown"

print(f"Running {len(TEST_INDICES)} images...\n")

for i, idx in enumerate(TEST_INDICES):
    sample = dataset[idx]
    image = sample["image"]
    gt_label = LABELS[idx]
    cleaned = CLEANED_CAPTIONS.get(idx, "N/A")
    
    # Convert image to base64
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    
    print(f"[{i+1}/{len(TEST_INDICES)}] Index {idx} (GT: {gt_label})")
    
    response = ollama.chat(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": PROMPT,
            "images": [img_b64]
        }],
        options={
            "temperature": SETTINGS["temperature"],
            "top_k": SETTINGS["top_k"],
            "top_p": SETTINGS["top_p"],
            "num_predict": SETTINGS["max_tokens"]
        }
    )
    
    generated = response["message"]["content"]
    pred = classify_response(generated)
    correct = (pred == gt_label)
    
    print(f"    Cleaned GT: {cleaned[:80]}...")
    print(f"    Response: {generated[:100]}...")
    print(f"    Predicted: {pred} | {'CORRECT' if correct else 'WRONG'}\n")
    
    results.append({
        "index": idx,
        "gt_label": gt_label,
        "cleaned_caption": cleaned,
        "response": generated,
        "pred_label": pred,
        "correct": correct
    })

print("=" * 50)
print("DONE")



## Results Summary

In [ ]:
correct = sum(1 for r in results if r["correct"])
total = len(results)

melanoma_results = [r for r in results if r["gt_label"] == "melanoma"]
nevus_results = [r for r in results if r["gt_label"] == "nevus"]

tp = sum(1 for r in melanoma_results if r["pred_label"] == "melanoma")
fn = sum(1 for r in melanoma_results if r["pred_label"] != "melanoma")
tn = sum(1 for r in nevus_results if r["pred_label"] == "nevus")
fp = sum(1 for r in nevus_results if r["pred_label"] == "melanoma")

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
balanced_acc = (sensitivity + specificity) / 2

print("RESULTS")
print("=" * 40)
print(f"Accuracy: {correct}/{total} ({100*correct/total:.1f}%)")
print(f"Sensitivity (melanoma recall): {sensitivity:.1%}")
print(f"Specificity (nevus recall): {specificity:.1%}")
print(f"Balanced Accuracy: {balanced_acc:.1%}")
print()
print(f"Prompt: {PROMPT}")
print(f"Settings: {SETTINGS}")


## Save Results to a file

In [ ]:
import json
from datetime import datetime
import os

# Save to current directory
filename = f"medgemma_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

save_data = {
    "prompt": PROMPT,
    "settings": SETTINGS,
    "accuracy": correct / total,
    "sensitivity": sensitivity,
    "specificity": specificity,
    "balanced_accuracy": balanced_acc,
    "results": results
}

with open(filename, "w") as f:
    json.dump(save_data, f, indent=2)

print(f"[SAVED] {os.path.abspath(filename)}")
